# **Import des librairies et chargement des données**

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
df_analyse = pd.read_csv("../data/2_interim/paquets_phrases.csv")
df_analyse.head()

,nom_fichier,id_paquet,phrases_paquet
0,1893_20_Le_docteur_Pascal._clean.txt,0,Dans la chaleur de l’ardente après-midi de jui...
1,1893_20_Le_docteur_Pascal._clean.txt,1,"Mais il dut prendre une chaise, la planche du ..."
2,1893_20_Le_docteur_Pascal._clean.txt,2,"Oh! Monsieur, la religion n’a jamais fait de m..."
3,1893_20_Le_docteur_Pascal._clean.txt,3,"De face, dans son visage séché, ses yeux garda..."
4,1893_20_Le_docteur_Pascal._clean.txt,4,s’écria la jeune fille. Mais elle était lancée...


# **Analyse préliminaire de la segmentation du corpus**

In [4]:
df_analyse.isna().sum()

nom_fichier       0
id_paquet         0
phrases_paquet    0
dtype: int64

In [5]:
df_analyse.duplicated(subset=["nom_fichier", "id_paquet"]).sum()

np.int64(0)

In [6]:
df_analyse.duplicated(subset=["phrases_paquet"]).sum()

np.int64(0)

- On constate que l'on a aucune valeur manquante dans les le jeu de données.

- Il n'y a pas non plus de doublons. 

## **affichage de la distribution du nombre de paquets par livres**

In [7]:
df_paquets = df_analyse.groupby("nom_fichier")["phrases_paquet"].count().reset_index()
df_paquets.columns = ["nom_fichier", "nb_paquets"]
df_paquets

,nom_fichier,nb_paquets
0,1865_La_confession_de_Claude._clean.txt,65
1,1866_Le_voeu_d_une_morte._clean.txt,60
2,1867_Les_mysteres_de_Marseille._clean.txt,184
3,1867_Therese_Raquin._clean.txt,80
4,1868_Madeleine_Ferat._clean.txt,124
5,1871_1_La_fortune_des_Rougon._clean.txt,154
6,1871_2_La_curee._clean.txt,123
7,1873_3_Le_ventre_de_Paris._clean.txt,136
8,1874_4_La_conquete_de_Plassans._clean.txt,165
9,1875_5_La_faute_de_l_abbe_Mouret._clean.txt,171


In [8]:
df_paquets.describe()

,nb_paquets
count,31.000000
mean,171.838710
std,52.395354
min,60.000000
25%,145.000000
50%,171.000000
75%,214.500000
max,259.000000


- Il y a un ecart type de 52. Cette écart est assez grand mais pas incohérent vu que les livres ont des tailles tres différentes. **Il y a un livre qui n'a que 60 paquets et un autre qui en a 259.**

- coefficient de variation est d’environ 30% ( écart-type / moyenne =>  52/171 ± 0.30) 

- La mediane est proche de la moyenne (171 pour la mediane et 172 pour la moyenne).

## **affichage de la distribution du nombre de mots par paquets**

In [9]:
df_phrases_mots = df_analyse["phrases_paquet"].str.split().str.len()
df_phrases_mots.describe()

count    5327.000000
mean      785.298667
std       212.129823
min        42.000000
25%       636.000000
50%       753.000000
75%       902.500000
max      2944.000000
Name: phrases_paquet, dtype: float64

- on constate qu'un paquet compte en moyenne 785 mots et la mediane compte 753 mots. Un nombre de 785 mots par paquet peut paraitre assez élévé pour du topic modeling, mais il faut garder à l'esprit que les paquets sont constitués de plusieurs phrases et que les livres de Zola sont souvent très longs. De plus, la segmentation en paquets de 5 phrases peut conduire à des paquets relativement bruités.

- l'ecart type est de 212,1. avec une dispesion **relativement faible 27%** (ecart type / moyenne => 212/785 ± 0.27).

- L’observation des quartiles indique que 25 % des paquets contiennent au maximum 636 mots, tandis que 75 % d’entre eux ne dépassent pas environ 903 mots. Malgré une certaine variabilité, la distribution reste relativement concentrée autour de la médiane, située à 753 mots. Cela est plutot un bon signe pour le topic modeling, car cela indique que la majorité des paquets ont une taille relativement homogène, ce qui peut faciliter l'identification de thèmes cohérents au sein des paquets.

- le seul point aberrant est un paquet qui contient 2944  mots, ce qui est assez élevé par rapport à la moyenne et à la médiane. Ainsi qu'un autre paquet qui ne contient seulemnt que 42 mots, ce qui est assez faible. Ce sont des valeurs extrêmes qui pourraient potentiellement influencer les résultats du topic modeling, il serait donc judicieux de les examiner de plus près pour comprendre leur nature et décider s'ils doivent être traités ou exclus de l'analyse.

## **analyse des valeurs extrêmes du corpus**

In [10]:
df_analyse["nb_mots_paquet"] = df_analyse["phrases_paquet"].str.split().str.len()
df_analyse

,nom_fichier,id_paquet,phrases_paquet,nb_mots_paquet
0,1893_20_Le_docteur_Pascal._clean.txt,0,Dans la chaleur de l’ardente après-midi de jui...,786
1,1893_20_Le_docteur_Pascal._clean.txt,1,"Mais il dut prendre une chaise, la planche du ...",940
2,1893_20_Le_docteur_Pascal._clean.txt,2,"Oh! Monsieur, la religion n’a jamais fait de m...",834
3,1893_20_Le_docteur_Pascal._clean.txt,3,"De face, dans son visage séché, ses yeux garda...",760
4,1893_20_Le_docteur_Pascal._clean.txt,4,s’écria la jeune fille. Mais elle était lancée...,854
...,...,...,...,...
5322,1894_1_Lourdes._clean.txt,193,"Le diable dans cette vie si pure, dans cette â...",937
5323,1894_1_Lourdes._clean.txt,194,Des milliers de pèlerins avaient beau se rendr...,983
5324,1894_1_Lourdes._clean.txt,195,"Elle était sa maîtresse souveraine, elle le te...",814
5325,1894_1_Lourdes._clean.txt,196,Et n’aurait-il pas fallu la venue d’un nouveau...,893


### **comptage des valeurs extrêmes**

In [31]:
print(f"il y'a ",(df_analyse["nb_mots_paquet"]<300).sum(), "paquets de phrases qui contiennent moins de 300 mots")

il y'a  12 paquets de phrases qui contiennent moins de 300 mots


In [32]:
print(f"il y'a ",(df_analyse["nb_mots_paquet"]>1200).sum(), "paquets de phrases qui contiennent plus de 1000 mots")

il y'a  245 paquets de phrases qui contiennent plus de 1000 mots


In [35]:
print(f"il y a ", ((df_analyse["nb_mots_paquet"]<300).mean().round(3))*100, "% de paquets de phrases qui contiennent moins de 300 mots")

il y a  0.2 % de paquets de phrases qui contiennent moins de 300 mots


In [34]:
print(f"il y a ", ((df_analyse["nb_mots_paquet"]>1200).mean().round(2))*100, "% de paquets de phrases qui contiennent plus de 1200 mots")

il y a  5.0 % de paquets de phrases qui contiennent plus de 1200 mots


### **inspection des segment de valeurs extrêmes**

#### 1) valeurs extrêmes minimum

In [14]:
df_analyse.sort_values("nb_mots_paquet").head(10)[
    ["nom_fichier", "id_paquet", "nb_mots_paquet", "phrases_paquet"]
]

,nom_fichier,id_paquet,nb_mots_paquet,phrases_paquet
3150,1884_12_La_joie_de_vivre._clean.txt,163,42,"Et ce misérable sans pieds ni mains, qu’il fal..."
3934,1873_3_Le_ventre_de_Paris._clean.txt,135,59,"Puis, toutes deux se penchèrent. La belle Mme ..."
3798,1866_Le_voeu_d_une_morte._clean.txt,59,62,"Il les tint ainsi serrées, jusqu’à ce que le s..."
2765,1878_8_Une_page_d_amour._clean.txt,165,132,"On sablait les rues, leur voiture ne mettrait ..."
4957,1891_18_L_argent._clean.txt,154,145,"au-dessus de tant de boue remuée, au-dessus de..."
834,1887_15_La_terre._clean.txt,221,157,"Et la terre seule demeure l’immortelle, la mèr..."
2311,1888_16_Le_reve._clean.txt,81,183,"Monseigneur, de son geste habituel de bénédict..."
3511,1890_17_La_bete_humaine._clean.txt,153,187,"Et la gare de Sotteville fut brûlée, il fila a..."
2986,1892_19_La_debacle._clean.txt,220,217,"Il lui sembla, dans cette lente tombée du jour..."
5326,1894_1_Lourdes._clean.txt,197,224,"C’était Paris dans sa forge, Paris avec ses pa..."


#### 2) valeurs extrêmes maximum

In [37]:
df_analyse.sort_values("nb_mots_paquet", ascending=False).head(15)[
    ["nom_fichier", "id_paquet", "nb_mots_paquet", "phrases_paquet"]
]

,nom_fichier,id_paquet,nb_mots_paquet,phrases_paquet
40,1893_20_Le_docteur_Pascal._clean.txt,40,2944,J’ai dû même spécifier un quatrième cas très r...
2110,1886_14_L_oeuvre._clean.txt,43,1845,"Quand la neige couvrait les toits voisins, que..."
4408,1896_2_Rome._clean.txt,61,1815,Et il évoqua ce qu’il savait de la splendeur d...
4825,1891_18_L_argent._clean.txt,22,1755,"Elle vaquait à ses occupations si multiples, m..."
4402,1896_2_Rome._clean.txt,55,1750,"Il s’était fait donner tous les titres, il ava..."
41,1893_20_Le_docteur_Pascal._clean.txt,41,1722,"Il était hors d’haleine, épuisé d’un tel souff..."
4431,1896_2_Rome._clean.txt,84,1709,"Une petite population s’y agite, travaille, ai..."
4863,1891_18_L_argent._clean.txt,60,1704,"Pour elle, dans ce dévidage du cœur et de la c..."
4479,1896_2_Rome._clean.txt,132,1691,"Et ils avaient, lui et ses frères, le verbe ha..."
4819,1891_18_L_argent._clean.txt,16,1641,"Massias sauta, avant même que le cocher eût ar..."


## **Tablau de répartition des paquets par livres avec moyenne et médiane**

In [17]:
repartition_romans = (
    df_analyse
    .groupby("nom_fichier")
    .agg(
        nb_paquets=("id_paquet", "count"),
        nb_mots_total=("nb_mots_paquet", "sum"),
        moyenne_mots_paquet=("nb_mots_paquet", "mean"),
        mediane_mots_paquet=("nb_mots_paquet", "median")
    )
    .round(2)
    .reset_index()
    .sort_values("nb_paquets", ascending=False)
)

repartition_romans

,nom_fichier,nb_paquets,nb_mots_total,moyenne_mots_paquet,mediane_mots_paquet
28,1899_1_Fecondite._clean.txt,259,220131,849.93,823.0
26,1896_2_Rome._clean.txt,231,233958,1012.81,1012.0
11,1877_7_L_assommoir._clean.txt,228,158340,694.47,679.5
30,1903_3_Verite._clean.txt,227,223274,983.59,948.0
13,1880_9_Nana._clean.txt,225,141573,629.21,607.0
17,1885_13_Germinal._clean.txt,225,164473,730.99,711.0
19,1887_15_La_terre._clean.txt,222,163687,737.33,706.0
23,1892_19_La_debacle._clean.txt,221,187046,846.36,832.0
29,1901_2_Travail._clean.txt,208,198712,955.35,944.5
14,1882_10_Pot-bouille._clean.txt,207,134844,651.42,631.0
